# 🚀 Hackathon Cloud GPU Runner (Kaggle / Colab)
### Free 2x Tesla T4 GPUs (32 GB Total VRAM)

**Team:** @Abhishekdabi33, @gunwantjawle-ux, @sarthakchambhare16-wdym, @vedxnt-b  
**Purpose:** Train, fine-tune, or serve open-weight **Google Gemma** and **Hermes** models in the cloud for free, with zero load on your local laptops.

---

## 1. Verify Cloud GPU Acceleration
Make sure **GPU T4 x2** is selected in Kaggle (Settings -> Accelerator -> GPU T4 x2) or Colab (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 2. Install Required AI & Agent Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate bitsandbytes peft datasets trl huggingface_hub pyngrok fastapi uvicorn
print("✅ Dependencies successfully installed!")

## 3. Load Google Gemma Model with 4-Bit Quantization (BitsAndBytes)
4-bit NF4 quantization reduces memory by 75%, allowing Gemma 9B to run seamlessly on a single T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

# Gemma is a gated model on Hugging Face:
# 1. Accept terms at: https://huggingface.co/google/gemma-2-2b-it (click Acknowledge License)
# 2. Get a free token at: https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # <-- Paste your token here

if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)

MODEL_ID = "google/gemma-2-2b-it"
# Tip: If you prefer zero logins, use: MODEL_ID = "NousResearch/Hermes-3-Llama-3.1-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"⏳ Loading {MODEL_ID} onto Kaggle GPU...")
token_arg = HF_TOKEN if HF_TOKEN != "YOUR_HF_TOKEN_HERE" else None
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token_arg)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=token_arg
)
print("🚀 Model successfully loaded onto Cloud GPU!")

## 4. Test Model Inference on Cloud GPU

In [ ]:
prompt = "Explain how autonomous AI agents use the SKILL.md open standard in two bullet points."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("🤖 Output:")
print(response)

## 5. Fine-Tuning Pipeline (QLoRA / PEFT)
Train custom weights on Kaggle GPUs and export them as lightweight LoRA adapters (`adapter_model.safetensors`).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print("📊 Trainable parameters:")
peft_model.print_trainable_parameters()

# Save trained adapter weights for your hackathon repository
peft_model.save_pretrained("./hackathon_gemma_lora")
print("💾 Checkpoint saved to ./hackathon_gemma_lora. Download this folder and place in ai_engine/checkpoints/")

## 6. (Optional) Expose Kaggle GPU as Live API Tunnel (ngrok)
If your team wants your local React web app to talk directly to this Kaggle GPU in real-time during the hackathon:
1. Get a free auth token from [ngrok.com](https://ngrok.com)
2. Run the cell below to get a public `https://....ngrok-free.app` URL!
3. Set `VITE_API_BASE_URL` or `CLOUD_GPU_URL` in your `.env` to this URL.

In [ ]:
import os
import threading
import uvicorn
from fastapi import FastAPI
from pyngrok import ngrok
from pydantic import BaseModel

# 1. Cleanly reset any old ngrok processes from earlier sessions
try:
    ngrok.kill()
except Exception:
    pass

# 2. Paste your free ngrok token below:
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

if NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    # Save auth token directly to config
    os.system(f"ngrok config add-authtoken {NGROK_TOKEN}")
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except Exception:
        pass

    # Open public HTTPS tunnel
    public_url = ngrok.connect(8000).public_url
    print(f"⚡ Public Cloud GPU URL: {public_url}")
    print("👉 Copy this URL into your local .env as CLOUD_GPU_URL!")

    app = FastAPI(title="Kaggle GPU Inference Server")

    class PromptReq(BaseModel):
        prompt: str
        max_tokens: int = 250

    @app.post("/api/predict")
    def predict(req: PromptReq):
        inputs = tokenizer(req.prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=req.max_tokens, do_sample=True, temperature=0.7)
        # Strip prompt: decode only new tokens
        new_tokens = out[0][inputs.input_ids.shape[1]:]
        clean_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not clean_text:
            clean_text = tokenizer.decode(out[0], skip_special_tokens=True)
        return {"result": clean_text, "device": "Kaggle Tesla T4 GPU"}

    threading.Thread(target=uvicorn.run, kwargs={"app": app, "host": "0.0.0.0", "port": 8000}, daemon=True).start()
else:
    print("ℹ️ Add your ngrok token above to expose a live public API from Kaggle.")